In [14]:
import illustris_python as il
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import defaultdict

In [15]:
# --- CONFIGURATION ---
basePath = '../sims.TNG/TNG100-1/output/'
csv_path = 'luminosity_check_data_tng_100.csv'
output_filename = 'processed_merger_data_with_ids_tng_100.csv'
BAND_INDEX = 5 # r-band

# --- SNAPSHOT FILTER ---
# Restrict processing to this range to match local data parity
MIN_SNAP = 68
MAX_SNAP = 99

# Load Input
df_input = pd.read_csv(csv_path)
target_group_ids = df_input['HaloID_Snap99'].unique()
print(f"Found {len(target_group_ids)} unique Groups to process.")

Found 1723 unique Groups to process.


In [16]:
# --- PHASE 1: PRE-CALCULATE MAPS ---
print("Loading Group Catalog for ID mapping...")
# We load the 'GroupFirstSub' field for all groups at Snap 99.
# This allows us to instantly convert Group ID -> Central Subhalo ID
group_cat_99 = il.groupcat.loadHalos(basePath, 99, fields=['GroupFirstSub'])

if isinstance(group_cat_99, dict):
    group_first_sub_map = group_cat_99['GroupFirstSub']
else:
    group_first_sub_map = group_cat_99

# Dictionary to hold the skeleton data: records_by_snap[snap_num] = list of requests
records_by_snap = defaultdict(list)

Loading Group Catalog for ID mapping...


In [17]:
# --- PHASE 2: TRACE TREES (Metadata Only) ---
print(f"Tracing Merger Trees (Snaps {MIN_SNAP}-{MAX_SNAP})...")

for group_id in tqdm(target_group_ids, desc="Tracing Trees"):
    # Safety check for bounds
    if group_id >= len(group_first_sub_map): continue
    
    central_sub_id = group_first_sub_map[group_id]
    
    # If -1, the group has no subhalos
    if central_sub_id == -1: continue

    # Load tree (Metadata only, fast)
    tree_fields = ['SnapNum', 'SubfindID', 'SubhaloGrNr', 'SubhaloMass']
    tree = il.sublink.loadTree(basePath, 99, central_sub_id, fields=tree_fields, onlyMPB=True)
    
    if tree is None or 'SnapNum' not in tree: continue
    
    count = len(tree['SnapNum'])
    
    for i in range(count):
        snap = tree['SnapNum'][i]
        
        # --- NEW FILTERING LOGIC ---
        # Skip if outside the desired range (maintains parity with local files)
        if snap < MIN_SNAP or snap > MAX_SNAP: 
            continue
        # ---------------------------

        central_id = tree['SubfindID'][i] # Global ID
        
        # We store the request for data
        records_by_snap[snap].append({
            'HaloID_Snap99': group_id,
            'Central_ID': central_id,
            'Tree_GrNr': tree['SubhaloGrNr'][i] # Store for safety check
        })

Tracing Merger Trees (Snaps 68-99)...


Tracing Trees: 100%|██████████| 1723/1723 [02:47<00:00, 10.27it/s]


In [18]:
# --- PHASE 3: BATCH PROCESS BY SNAPSHOT ---
final_history = []

sorted_snaps = sorted(records_by_snap.keys())

for snap in tqdm(sorted_snaps, desc="Processing Snapshots"):
    requests = records_by_snap[snap]
    if not requests: continue
    
    # 1. Load Group Catalog for this snap (To get N_subs and First_Sub)
    try:
        group_cat = il.groupcat.loadHalos(basePath, snap, fields=['GroupFirstSub', 'GroupNsubs'])
    except Exception as e:
        print(f"Error loading group cat for snap {snap}: {e}")
        continue
    
    # 2. Load ENTIRE Photometry Catalog for this snap
    # TNG uses 'SubhaloStellarPhotometrics'
    try:
        sub_mags_all = il.groupcat.loadSubhalos(basePath, snap, fields=['SubhaloStellarPhotometrics'])
    except Exception as e:
        print(f"Skipping Snap {snap}: {e}")
        continue
    
    # Check if load returned correctly (dict vs array)
    if isinstance(sub_mags_all, dict):
        if 'SubhaloStellarPhotometrics' in sub_mags_all:
            sub_mags_all = sub_mags_all['SubhaloStellarPhotometrics']
        elif 'SubhaloPhotometrics' in sub_mags_all:
            sub_mags_all = sub_mags_all['SubhaloPhotometrics']
    
    # Pre-select r-band to save memory/time in loop
    r_band_all = sub_mags_all[:, BAND_INDEX]
    
    # 3. Process requests for this snap
    for req in requests:
        group_id = req['Tree_GrNr']
        central_id = req['Central_ID'] # Global ID
        
        # Safety
        if group_id >= len(group_cat['GroupFirstSub']): continue
        
        first_sub = group_cat['GroupFirstSub'][group_id]
        n_subs = group_cat['GroupNsubs'][group_id]
        
        if n_subs <= 1:
            # Empty group or just central
            continue
            
        # Determine local index of central to mask it
        local_central_idx = central_id - first_sub
        
        # Validations
        if local_central_idx < 0 or local_central_idx >= n_subs:
            continue # Tree mismatch
            
        # --- EXTRACT MAGNITUDES ---
        # Ensure we don't go out of bounds of the subhalo catalog
        if first_sub + n_subs > len(r_band_all): continue

        group_mags = r_band_all[first_sub : first_sub + n_subs]
        
        central_mag = group_mags[local_central_idx]
        
        # Mask central to find brightest sat
        mask = np.ones(n_subs, dtype=bool)
        mask[local_central_idx] = False
        
        sat_mags = group_mags[mask]
        
        if len(sat_mags) == 0:
            continue
            
        # Find Brightest
        brightest_idx_local = np.argmin(sat_mags)
        sat_mag = sat_mags[brightest_idx_local]
        
        # Map back to Global ID
        # We need the global index corresponding to the brightest satellite
        # Re-construct global IDs for this group
        group_global_ids = np.arange(first_sub, first_sub + n_subs)
        sat_global_ids = group_global_ids[mask]
        sat_id = sat_global_ids[brightest_idx_local]
        
        # Store Result
        final_history.append({
            'HaloID_Snap99': req['HaloID_Snap99'],
            'SnapNum': snap,
            'Mag_Central_r': central_mag,
            'Brightest_Sat_Mag_r': sat_mag,
            'Sat_ID': sat_id,
            'Mag_Gap': sat_mag - central_mag
        })

Processing Snapshots: 100%|██████████| 32/32 [19:09<00:00, 35.91s/it]


In [19]:
# --- PHASE 4: ANALYZE & EXPORT ---
print("Analyzing Lineages...")
df_all = pd.DataFrame(final_history)

# Helper function to check lineage connectivity using SubLink (Inverted Logic)
def are_related(basePath, ancestor_id, ancestor_snap, descendant_id, descendant_snap):
    """
    Returns True if 'ancestor_id' at 'ancestor_snap' is a 
    PROGENITOR of 'descendant_id' at 'descendant_snap'.
    """
    try:
        # Load the PROGENITOR tree for the FUTURE subhalo (standard behavior)
        # We look backwards from the recovery point
        tree = il.sublink.loadTree(basePath, descendant_snap, descendant_id, 
                                   fields=['SnapNum', 'SubfindID'], 
                                   onlyMPB=True)
        
        if tree is None: return False
        
        # Look for the ancestor snapshot in this tree
        # tree['SnapNum'] contains the history (past) of the subhalo
        match_idx = np.where(tree['SnapNum'] == ancestor_snap)[0]
        
        if len(match_idx) > 0:
            # Get the ID the descendant had at that earlier snapshot
            past_id_from_tree = tree['SubfindID'][match_idx[0]]
            
            # Does it match the ID of the satellite we saw at the dip?
            return past_id_from_tree == ancestor_id
            
    except Exception as e:
        print(e)
        return False
        
    return False

def analyze_events_for_group(df_group, basePath):
    # Sort by SnapNum (Low -> High) to ensure time order
    df_group = df_group.sort_values('SnapNum')
    
    gaps = df_group['Mag_Gap'].values
    ids = df_group['Sat_ID'].values
    snaps = df_group['SnapNum'].values
    
    # If history is too short in this window (less than ~3 snaps), we can't reliably classify
    if len(gaps) < 3: return "Stable"
    
    min_gap = np.min(gaps)
    
    # 1. Stripping Check (Gap widens significantly from start to end of window)
    if (gaps[-1] - gaps[0]) > 1.5:
        return "Stripping"
        
    # 2. Interaction Check (Fly-by vs Infall)
    GAP_THRESHOLD = 1.5
    
    if min_gap < GAP_THRESHOLD:
        # Find the point of closest approach (the dip)
        min_idx = np.argmin(gaps)
        sat_id_at_min = ids[min_idx]
        snap_at_min = snaps[min_idx]
        
        # Look at the future relative to the dip
        future_gaps = gaps[min_idx:]
        future_ids = ids[min_idx:]
        future_snaps = snaps[min_idx:]
        
        # Need at least one snapshot after the dip to check recovery
        if len(future_gaps) > 1:
            max_future_gap = np.max(future_gaps)
            
            # Did the gap ever open up again?
            if max_future_gap > GAP_THRESHOLD:
                # Find the FIRST point where it recovered
                recovery_indices = np.where(future_gaps > GAP_THRESHOLD)[0]
                
                if len(recovery_indices) > 0:
                    first_recovery_idx = recovery_indices[0]
                    
                    id_at_recovery = future_ids[first_recovery_idx]
                    snap_at_recovery = future_snaps[first_recovery_idx]
                    
                    # --- LINEAGE CHECK (INVERTED) ---
                    # Check if the satellite at recovery is the future self of the one at the dip
                    if are_related(basePath, sat_id_at_min, snap_at_min, id_at_recovery, snap_at_recovery):
                        return "Fly-by"
                    else:
                        # The original satellite was lost/merged, and this is a new one
                        return "Infall"
            else:
                # Gap dropped and stayed low forever (within this window)
                return "Infall"
        else:
            # Dip happened at the very last snapshot, assume Infall started
            return "Infall"
            
    return "Stable"

# --- EXECUTE CLASSIFICATION ---
results = []
if not df_all.empty:
    unique_halos = df_all['HaloID_Snap99'].unique()
    
    # Iterate with progress bar
    for hid in tqdm(unique_halos, desc="Classifying Events"):
        group = df_all[df_all['HaloID_Snap99'] == hid]
        
        # Pass basePath so the function can query the Merger Trees
        cat = analyze_events_for_group(group, basePath)
        
        # Store result
        group = group.copy()
        group['Merger_Class'] = cat
        results.append(group)

    # Save final CSV
    final_df = pd.concat(results)
    final_df.to_csv(output_filename, index=False)
    print(f"Done! Saved analysis to {output_filename}")
    
    # Print sample stats
    print("\nClassification Counts:")
    print(final_df[['HaloID_Snap99', 'Merger_Class']].drop_duplicates()['Merger_Class'].value_counts())
else:
    print("No results generated.")

Analyzing Lineages...


Classifying Events:  20%|██        | 349/1723 [00:15<00:46, 29.68it/s]

Warning, empty return. Subhalo [359908] at snapNum [74] not in tree.


Classifying Events:  22%|██▏       | 379/1723 [00:16<00:58, 23.00it/s]

Warning, empty return. Subhalo [393865] at snapNum [82] not in tree.


...

Warning, empty return. Subhalo [492345] at snapNum [92] not in tree.


Classifying Events:  97%|█████████▋| 1677/1723 [00:48<00:00, 76.83it/s] 

Warning, empty return. Subhalo [527355] at snapNum [99] not in tree.


Classifying Events: 100%|██████████| 1723/1723 [00:49<00:00, 35.11it/s]


Done! Saved analysis to processed_merger_data_with_ids_tng_100.csv

Classification Counts:
Merger_Class
Stable       717
Infall       427
Stripping    426
Fly-by       153
Name: count, dtype: int64
